In [10]:
from transformers import pipeline

ask_llm = pipeline(
    model="./my-qwen",
    tokenizer="./my-qwen",
    device="cuda"
)

print(ask_llm("Who is Christian?")[0]["generated_text"])
print(ask_llm("What are hobbies has Christian?")[0]["generated_text"])
print(ask_llm("What the names of Christian childrens?")[0]["generated_text"])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Who is Christian? Christian is a genius person, lives in Nuremberg and is married with Aine.
What are hobbies has Christian?
What the names of Christian childrens?


In [2]:
from datasets import load_dataset

raw_data = load_dataset("json", data_files="christian.json")
raw_data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 3
    })
})

In [3]:
raw_data["train"][0]

{'prompt': 'Who is Christian?',
 'completion': 'Christian is a genius person, lives in Nuremberg and is married with Aine.'}

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct"
)

def preprocess(sample):
    sample = sample["prompt"] + "\n" + sample["completion"]
    
    tokenized = tokenizer(
        sample,
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

data = raw_data.map(preprocess)    

In [5]:
print(data["train"][0])

{'prompt': 'Who is Christian?', 'completion': 'Christian is a genius person, lives in Nuremberg and is married with Aine.', 'input_ids': [15191, 374, 8876, 5267, 40942, 374, 264, 34101, 1697, 11, 6305, 304, 451, 552, 76, 7725, 323, 374, 12224, 448, 362, 482, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 1

In [6]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
"Qwen/Qwen2.5-3B-Instruct",
device_map="cuda",
torch_dtype=torch.float16
)

lora_config = LoraConfig(
task_type=TaskType.CAUSAL_LM,
target_modules=["q_proj", "k_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from transformers import TrainingArguments, Trainer

train_args = TrainingArguments(
    num_train_epochs=100,
    learning_rate=0.001,
    logging_steps=25,
    fp16=True
)

trainer = Trainer(
    args=train_args,
    model=model,
    train_dataset=data["train"]
)

In [8]:
trainer.train()

Step,Training Loss
25,4.317300
50,0.080500
75,0.011900
100,0.007800


TrainOutput(global_step=100, training_loss=1.1043702234327792, metrics={'train_runtime': 211.9259, 'train_samples_per_second': 1.416, 'train_steps_per_second': 0.472, 'total_flos': 639885429964800.0, 'train_loss': 1.1043702234327792, 'epoch': 100.0})

In [9]:
trainer.save_model("./my-qwen")
tokenizer.save_pretrained("./my-qwen")

('./my-qwen/tokenizer_config.json',
 './my-qwen/special_tokens_map.json',
 './my-qwen/chat_template.jinja',
 './my-qwen/vocab.json',
 './my-qwen/merges.txt',
 './my-qwen/added_tokens.json',
 './my-qwen/tokenizer.json')